# ⚠️ Notebook histórico — superado

Este notebook conserva evidencia de la primera comprobación. **No es el entregable final.**

La reserva Dataset B v1 usaba el split `train` de FLEURS y fue reemplazada por el split oficial `test`:

`data/processed/manifests/fleurs_evaluation_v2_official_test.csv`

Use `notebooks/Proyecto_IA_Traductor_MASTER_COLAB.ipynb` para la ejecución final.


# Comprobación formal — Dataset B: FLEURS es_419 Speech-to-Speech

Notebook autónomo para Google Colab. Reconstruye la reserva versionada y procesa exclusivamente sus seis filas test con MarianMT base.

# 1. Inicialización

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Setrox1250/Proyecto-IA-Traductor.git"
REPO = Path("/content/Proyecto-IA-Traductor")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    if not (REPO / ".git").exists():
        raise RuntimeError("La ruta existe pero no es un repositorio Git.")
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True
    ).strip()
    branch_existing = subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"], text=True
    ).strip()
    if not branch_existing:
        raise RuntimeError("El clon está en detached HEAD.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "--all", "--prune"], check=True)
    if dirty:
        print("El clon contiene trabajo local; se conserva HEAD y no se ejecuta pull.")
    else:
        subprocess.run(
            ["git", "-C", str(REPO), "pull", "--ff-only", "origin", branch_existing],
            check=True,
        )
os.chdir(REPO)
branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
last_commit = subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip()
print("Rama:", branch)
print("Commit:", commit)
print("Último commit:", last_commit)

# 2. Entorno

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        "datasets>=3.0.0", "sacrebleu>=2.4.0", "jiwer>=3.0.0",
        "soundfile>=0.12.0", "scipy>=1.11.0", "matplotlib>=3.7.0",
        "sentencepiece>=0.1.99", "sacremoses>=0.1.1",
        "transformers==4.57.3", "coqui-tts==0.27.5", "speechbrain==1.1.1",
    ],
    check=True,
)
os.environ["COQUI_TOS_AGREED"] = "1"

import json
import platform
import time
import importlib.metadata as metadata
import torch
import torchaudio
import transformers
import datasets
import numpy as np

def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

OUT = Path("outputs/comprobacion/datasetB")
OUT.mkdir(parents=True, exist_ok=True)
environment = {
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "python": sys.version,
    "operating_system": platform.platform(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "vram_total_gib": (
        round(torch.cuda.get_device_properties(0).total_memory / 2**30, 3)
        if torch.cuda.is_available() else None
    ),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "torchaudio": torchaudio.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "sacrebleu": package_version("sacrebleu"),
    "jiwer": package_version("jiwer"),
    "coqui_tts": package_version("coqui-tts"),
    "speechbrain": package_version("speechbrain"),
    "numpy": np.__version__,
    "git_branch": branch,
    "git_commit": commit,
    "git_log_1": last_commit,
}
(OUT / "environment.json").write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(environment, ensure_ascii=False, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError("Seleccione un runtime con GPU antes de continuar.")

# 3. Dataset B

La ejecución reconstruye los 20 identificadores fijados en el manifiesto del repositorio y se detiene si no obtiene 14 dev y 6 test.

# 4. Test reservado

Solo se procesan las seis filas test; no se eligen sustitutos ni muestras aleatorias.

# 5. ASR

Whisper-small se ejecuta en modo transcripción española. Se calculan WER raw, WER normalizado y CER.

# 6. NMT

La evaluación obligatoria utiliza Helsinki-NLP/opus-mt-es-en base con max_length=128 y num_beams=4.

# 7. XTTS

XTTS-v2 genera output_en.wav usando la voz del input como referencia.

# 8. ECAPA

ECAPA-TDNN calcula el coseno continuo original versus salida, sin convertirlo a porcentaje.

# 9. Pipeline

In [ ]:
# Esta celda ejecuta reconstrucción, ASR, NMT base, XTTS, ECAPA,
# métricas, artefactos, figuras y reporte. La primera muestra incluye cold start.
run_globals = {"__name__": "__main__"}
exec(
    compile(
        Path("scripts/run_comprobacion_dataset_b.py").read_text(encoding="utf-8"),
        "scripts/run_comprobacion_dataset_b.py",
        "exec",
    ),
    run_globals,
)

# 10. Métricas

In [ ]:
import pandas as pd
aggregate_metrics = json.loads(
    (OUT / "aggregate_metrics.json").read_text(encoding="utf-8")
)
results_per_sample = pd.read_csv(OUT / "results_per_sample.csv")
display(pd.DataFrame([aggregate_metrics]))
display(results_per_sample)

# 11. Análisis de errores

In [ ]:
error_file = OUT / "error_propagation_examples.csv"
error_examples = pd.read_csv(error_file)
if error_examples.empty:
    print("No se detectaron diferencias ortográficas ASR en las seis muestras.")
else:
    display(error_examples)

# 12. Exportación

In [ ]:
zip_path = Path("/content/comprobacion_5_1_artifacts.zip")
if not zip_path.exists():
    raise RuntimeError("No se generó el ZIP final.")
print("ZIP final:", zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Fuera de Colab: descargue el ZIP manualmente.")